# Gemma-4-E4B-it



In [ ]:
%pip install -q \
  torch==2.13.0 \
  transformers==5.16.1 \
  datasets==5.0.1 \
  accelerate==1.14.0 \
  peft==0.20.0 \
  trl==1.12.0 \
  bitsandbytes==0.50.2 \
  evaluate==0.4.6 \
  requests tqdm sentencepiece huggingface_hub pandas scikit-learn


In [ ]:
import json, random, re, string, time
from pathlib import Path
from collections import Counter
import numpy as np
import torch
from datasets import Dataset, DatasetDict

SEED=42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DATA_REPO="https://github.com/Gokcimen/Home_Appliance_Dataset"
!rm -rf /content/Home_Appliance_Dataset
!git clone -q --depth 1 {DATA_REPO}.git /content/Home_Appliance_Dataset

DATA_ROOT=Path("/content/Home_Appliance_Dataset")

def flatten(path):
    raw=json.loads(Path(path).read_text(encoding="utf-8"))
    rows=[]
    for article in raw["data"]:
        title=article["title"]
        for para in article["paragraphs"]:
            context=para["context"]
            for qa in para["qas"]:
                rows.append({
                    "id":str(qa["id"]),
                    "title":title,
                    "context":context,
                    "question":qa["question"],
                    "answers":{
                        "text":[a["text"] for a in qa["answers"]],
                        "answer_start":[int(a["answer_start"]) for a in qa["answers"]],
                    }
                })
    return rows

raw_datasets=DatasetDict({
    "train":Dataset.from_list(flatten(DATA_ROOT/"train.json")),
    "validation":Dataset.from_list(flatten(DATA_ROOT/"dev.json")),
    "test":Dataset.from_list(flatten(DATA_ROOT/"test.json")),
})

assert len(raw_datasets["train"])==8000
assert len(raw_datasets["validation"])==1000
assert len(raw_datasets["test"])==1000

ids={s:set(raw_datasets[s]["id"]) for s in raw_datasets}
assert not ids["train"]&ids["validation"]
assert not ids["train"]&ids["test"]
assert not ids["validation"]&ids["test"]
all_ids=set().union(*ids.values())
assert len(all_ids)==10000
assert {int(x) for x in all_ids}==set(range(1,10001))

titles={s:set(raw_datasets[s]["title"]) for s in raw_datasets}
assert not titles["train"]&titles["validation"]
assert not titles["train"]&titles["test"]
assert not titles["validation"]&titles["test"]
assert len(set().union(*titles.values()))==1111

print("train",len(raw_datasets["train"]))
print("validation",len(raw_datasets["validation"]))
print("test",len(raw_datasets["test"]))
print("products",len(set().union(*titles.values())))


In [ ]:
MODEL_KEY="gemma4"
MODEL_ID="google/gemma-4-E4B-it"
print(MODEL_ID)

In [ ]:
SYSTEM_PROMPT=(
    "You are a precise question-answering assistant for home-appliance information. "
    "Use only the supplied context for factual claims. "
    "Return only the answer."
)

def to_sft(ex):
    gold=ex["answers"]["text"][0]
    return {
        "text":(
            f"System: {SYSTEM_PROMPT}\n"
            f"Context:\n{ex['context']}\n\n"
            f"Question:\n{ex['question']}\n\n"
            f"Answer: {gold}"
        )
    }

train_sft=raw_datasets["train"].map(
    to_sft,remove_columns=raw_datasets["train"].column_names
)
validation_sft=raw_datasets["validation"].map(
    to_sft,remove_columns=raw_datasets["validation"].column_names
)


In [ ]:
from huggingface_hub import notebook_login
notebook_login()

import torch
from transformers import AutoProcessor, AutoModelForMultimodalLM, BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTConfig, SFTTrainer

processor=AutoProcessor.from_pretrained(MODEL_ID)
tokenizer=processor.tokenizer
if tokenizer.pad_token is None:
    tokenizer.pad_token=tokenizer.eos_token

bnb=BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

model=AutoModelForMultimodalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)
model=prepare_model_for_kbit_training(model)

peft_config=LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    target_modules="all-linear",
)

OUTPUT_DIR=f"/content/outputs/{MODEL_KEY}"

args=SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    learning_rate=3e-5,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,
    weight_decay=0.01,
    lr_scheduler_type="linear",
    warmup_ratio=0.10,
    max_grad_norm=1.0,
    max_length=1024,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    seed=42,
    data_seed=42,
    report_to="none",
)

trainer=SFTTrainer(
    model=model,
    args=args,
    train_dataset=train_sft,
    eval_dataset=validation_sft,
    processing_class=tokenizer,
    peft_config=peft_config,
)
trainer.train()
trainer.save_model(f"{OUTPUT_DIR}/final_adapter")


In [ ]:
def normalize_answer(text):
    text=str(text).lower()
    text="".join(c for c in text if c not in string.punctuation)
    text=re.sub(r"\b(a|an|the)\b"," ",text)
    return " ".join(text.split())

def exact_match(pred,gold):
    return float(normalize_answer(pred)==normalize_answer(gold))

def token_f1(pred,gold):
    p=normalize_answer(pred).split()
    g=normalize_answer(gold).split()
    if not p or not g:
        return float(p==g)
    same=sum((Counter(p)&Counter(g)).values())
    if same==0:
        return 0.0
    precision=same/len(p)
    recall=same/len(g)
    return 2*precision*recall/(precision+recall)

def score_rows(rows):
    return {
        "n":len(rows),
        "EM":100*sum(exact_match(r["prediction"],r["gold"]) for r in rows)/len(rows),
        "F1":100*sum(token_f1(r["prediction"],r["gold"]) for r in rows)/len(rows),
        "mean_latency_seconds":sum(float(r.get("latency_seconds",0)) for r in rows)/len(rows),
    }


In [ ]:
import json, time, glob, pandas as pd
from peft import PeftModel

def generate_answer(model,tokenizer,ex):
    prompt=(
        f"{SYSTEM_PROMPT}\n\n"
        f"Context:\n{ex['context']}\n\n"
        f"Question:\n{ex['question']}\n\n"
        "Answer:"
    )
    inputs=tokenizer(prompt,return_tensors="pt",truncation=True,max_length=1024).to(model.device)
    with torch.inference_mode():
        output=model.generate(
            **inputs,
            max_new_tokens=64,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    pred=tokenizer.decode(
        output[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True,
    ).strip()
    return pred.splitlines()[0].strip() if pred else ""

rows=[]
for ex in raw_datasets["test"]:
    t0=time.perf_counter()
    pred=generate_answer(model,tokenizer,ex)
    rows.append({
        "id":ex["id"],
        "prediction":pred,
        "gold":ex["answers"]["text"][0],
        "latency_seconds":time.perf_counter()-t0,
    })

metrics=score_rows(rows)
print(metrics)

with open(f"{OUTPUT_DIR}/test_predictions.jsonl","w",encoding="utf-8") as f:
    for row in rows:
        f.write(json.dumps(row,ensure_ascii=False)+"\n")
with open(f"{OUTPUT_DIR}/test_metrics.json","w",encoding="utf-8") as f:
    json.dump(metrics,f,indent=2)
